# NB_EXTRACT_3RD_PARTY_CDC_BRZ_TO_SLV

Extract tài liệu crawl 3rd-party từ `lh_vv_bronze.dbo.poi_raw_event` vào `lh_vv_silver.dbo.slv_3p_poi_*` (03/10/2026).
Chỉ extract: đưa payload về bảng theo đúng cấu trúc, chuẩn kiểu và khoá. Không rule nghiệp vụ, không canonical, không gate.
Bảng, cột, đường dẫn, kiểu, khoá, cách tách dòng đọc từ bảng control `lh_vv_bronze.ctrl`;
hàm dùng chung (đọc VERSION, khoá, state, reject, log, watermark, wave) lấy từ `NB_LIB_EXTRACT_RAWDATA` (`%run`).

| | |
|---|---|
| Gọi bởi | Không nằm trong `PL_VV_TRANSFORM_BRONZE_TO_SILVER_FL_00`. Pipeline này không có Switch và chỉ gọi notebook partner. Notebook 3rd-party chạy tay hoặc bằng script ngoài pipeline |
| Đọc | `ctrl_mng_pipeline_config` (`load_mode`, `align_path`, `dedup_order`), `ctrl_cfg_schema_registry`, `ctrl_mng_watermark`, `ctrl_cdc_state`; raw: file mới theo `_delta_log` (VERSION) hoặc toàn bộ (FULL) |
| Ghi | `slv_3p_poi_*` (MERGE, không xoá), `ctrl_cdc_state`, `ctrl_cdc_reject`, `ctrl_log_run`, `ctrl_log_table_run`, `ctrl_mng_watermark` |
| Chạy song song | Bảng active cùng priority 1 → 1 wave, `max_parallel` luồng ([04/10] 14 bảng cấu hình, 13 active: `slv_3p_poi_policy` tắt) |
| Chạy lại | An toàn: state (`poi_id`, hoặc khoá bảng nếu `state_key = "row"`) bỏ tài liệu đã áp dụng; MERGE chỉ ghi đè khi `(_crawled_at, _event_id)` mới ≥ đang lưu; watermark chỉ tiến khi mọi bảng thành công |
| Chạy chồng | Khoá trên dòng watermark `wm_transform_poi_raw_event` → run thứ 2 `SKIPPED_CONCURRENT`. Chạy song song với partner: MERGE `ctrl_cdc_state` / `ctrl_cdc_reject` chỉ đọc partition `src_tbl` của mình → không xung đột |

**Khác partner** (chi tiết: `claude/POI_3P_EXTRACT_REDESIGN.md` §5):

| | Partner (CDC) | 3rd-party (snapshot) |
|---|---|---|
| 1 event | 1 thay đổi dòng của 1 bảng Postgres | 1 tài liệu đầy đủ của 1 địa điểm lúc crawl |
| Bảng nhận event | Đúng 1 bảng (`source.table`) | Mọi bảng; bảng nhận khi tài liệu có khối của bảng |
| Thứ tự | `(ts_ms, lsn, ingest)` | `crawled_at`; trùng thì `event_id` |
| State | Theo khoá bảng | Mặc định theo `poi_id` cho mọi bảng (đơn vị là tài liệu); `state_key = "row"`: theo khoá bảng |
| Khối / trường vắng | TOAST | Khối vắng (JSON null / không có) → bảng không nhận tài liệu đó, giữ dữ liệu cũ |
| Xoá | `op = d` | Không có; bảng con chỉ upsert, `_last_seen_at` cho biết lần cuối thấy |
| Khoá | Có sẵn trong payload | Suy ra: `poi_id` (md5), `review_id` (sha256), `media_dedup_key` (sha256) — `convert_rule` `HASH_*` |

Luồng: nạp + kiểm tra cấu hình → nhận khoá, mở run log → chốt version raw → đọc phần raw mới 1 lần → phân loại tài liệu
VALID / IGNORED / REJECTED, parse payload 1 lần → mỗi bảng: tách dòng theo `load_mode` → chuyển kiểu, khoá → lọc tài liệu cũ theo state
→ chọn bản mới nhất mỗi khoá → MERGE → commit state, reject, log, watermark → exit JSON.


In [1]:
# =============================================================================
# THAM SỐ (toggle "parameters"; pipeline truyền đè)
# =============================================================================
pl_name                 = "PL_VV_TRANSFORM_BRONZE_TO_SILVER_FL_00"   # @pipeline().Pipeline
run_id                  = ""        # @pipeline().RunId; để trống khi chạy tay
src_schema              = "lh_vv_bronze.dbo"
src_tbl                 = "poi_raw_event"
allow_full_scan         = True     # True: cho phép đọc FULL khi không đọc theo version được (lần đầu, sau reset, thiếu log, raw bị tạo lại)
max_parallel            = 12         # số bảng chạy đồng thời (14 bảng cùng 1 wave)
max_retries             = 1         # thử lại 1 bảng khi gặp lỗi tạm thời (xung đột ghi Delta...)
table_filter            = ""        # vd "slv_3p_poi_review,slv_3p_poi_media"; rỗng = tất cả bảng active
full_reload             = False     # True: đọc toàn bộ raw, bỏ qua state (nên dùng kèm table_filter)
dry_run                 = False     # True: chỉ đọc, parse, đếm (kể cả số giá trị thành NULL khi chuyển kiểu); không ghi gì
stop_on_failure         = False     # True: wave có bảng lỗi thì không chạy các wave sau
cast_null_policy        = "FAIL"    # FAIL: bảng có giá trị thành NULL khi chuyển kiểu -> FAILED trước MERGE, giữ watermark ; WARN: chỉ cảnh báo
running_timeout_minutes = 780       # [SỬA 08/10] 13 giờ, hơn 12 giờ; phải lớn hơn thời gian chạy dài nhất, kể cả lần FULL
sources                 = "google"  # source_name được xử lý (phẩy ngăn cách, không phân biệt hoa thường); rỗng = mọi nguồn. Ngoài danh sách -> IGNORED
langs                   = "vi,en,ko"  # ngôn ngữ nhận cho bảng theo ngôn ngữ (LANG, LANG_ARRAY); rỗng = mọi ngôn ngữ. Ngôn ngữ khác: đếm, bỏ qua
lang_object_path        = "extra_info.enrichment"          # object bản địa hoá chính trong payload; ngôn ngữ ở trường lang_field
lang_field              = "lang"
lang_map_path           = "extra_info.enrichmentSiblings"  # map ngôn ngữ -> object bản địa hoá
doc_key_column          = "poi_id"  # cột (có trong registry của MỌI bảng) định danh tài liệu
state_key               = "doc"     # entity_key của ctrl_cdc_state: "doc" = doc_key_column (mọi bảng) ; "row" = khoá bảng (bảng con nhận cả phần tử chỉ có ở tài liệu cũ đến trễ, state nhiều dòng hơn ~3 lần)

StatementMeta(, 611ec303-fa15-471c-a8d4-04a943aaf370, 3, Finished, Available, Finished, False)

In [2]:
%run NB_LIB_EXTRACT_RAWDATA

StatementMeta(, 611ec303-fa15-471c-a8d4-04a943aaf370, 17, Finished, Available, Finished, True)

## Hằng số nguồn `poi_raw_event`

| Hằng số | Ý nghĩa |
|---|---|
| `RAW_PAYLOAD` | Cột chứa tài liệu JSON |
| `RAW_EVENT_ID` | Mã event; phân định 2 tài liệu cùng địa điểm cùng `crawled_at` |
| `RAW_DOC_ID` | Cột định danh địa điểm ở nguồn, bắt buộc có giá trị (thiếu → REJECTED). Cột đầu là tên nguồn, lọc theo tham số `sources` |
| `RAW_LANG` | Ngôn ngữ gốc của tài liệu, gộp vào `_langs` |
| `IGNORE_REASONS` | Lý do bỏ qua (chỉ đếm, không phải lỗi) |
| `ENTRY_TYPE` | Kiểu mảng object bản địa hoá: `array<struct<lang, obj>>` |

Cột thứ tự là `watermark_column` của dòng watermark (`crawled_at`), không viết cứng ở đây.
Cột raw khác cần cho bảng đích khai báo trong registry dạng `_doc.<cột>`; notebook tự đọc thêm.

In [3]:
# =============================================================================
# HẰNG SỐ NGUỒN + CONTEXT NOTEBOOK
# =============================================================================
import uuid

from pyspark.sql.window import Window

RAW_PAYLOAD    = "normalized_payload"
RAW_EVENT_ID   = "event_id"
RAW_DOC_ID     = ("source_name", "source_id")
RAW_LANG       = "language_code"
IGNORE_REASONS = ("OUT_OF_SCOPE",)
ENTRY_TYPE     = "array<struct<lang:string,obj:string>>"
ARRAY_MODES    = ("DOC_ARRAY", "LANG_ARRAY")
LANG_MODES     = ("LANG", "LANG_ARRAY")

try:
    NB_NAME = notebookutils.runtime.context.get("currentNotebookName") or "NB_EXTRACT_3RD_PARTY_CDC_BRZ_TO_SLV"
except Exception:
    NB_NAME = "NB_EXTRACT_3RD_PARTY_CDC_BRZ_TO_SLV"

StatementMeta(, 611ec303-fa15-471c-a8d4-04a943aaf370, 18, Finished, Available, Finished, False)

## A. Đọc raw, phân loại và dựng tài liệu

| Hàm | Mục đích |
|---|---|
| `path_value(parsed, path)` | Lấy `a.b.c` từ struct đã parse: trường đầu từ struct, phần sau bằng `get_json_object` |
| `block_col(field)` / `doc_value(path)` | Sau phân loại, mỗi trường cấp 1 của payload là 1 cột `_b__<trường>` (cache đọc đúng cột cần, không giải mã cả payload cho mỗi bảng); `doc_value` lấy đường dẫn tính từ gốc payload |
| `spec_paths(spec)` | Mọi đường dẫn (đã tách phần) của các cột 1 bảng |
| `top_fields(specs)` / `doc_columns(specs)` | Trường cấp 1 của payload / cột raw (`_doc.*`) cần đọc cho các bảng trong phạm vi |
| `uses_langs(specs)` | Có bảng nào cần object bản địa hoá hoặc `_langs` không |
| `read_raw(ctx, doc_cols)` | Đọc phần raw mới bằng `read_incremental` của thư viện (VERSION / FULL) |
| `lang_entries(p)` | Mảng `(lang, obj)` từ `lang_object_path` + `lang_map_path`; trùng ngôn ngữ thì object chính thắng |
| `classify_docs(raw, fields, need_langs)` | Parse payload 1 lần; gắn `_status` VALID / IGNORED / REJECTED, `_reason`; dựng cột event (thứ tự) và ngôn ngữ |
| `presence(spec)` | Điều kiện tài liệu có khối của bảng (để bảng không có khối nào → NO_DATA) |
| `profile_docs(classified, specs, need_langs)` | Đếm theo trạng thái/lý do/nguồn, `watermark_to`, số tài liệu có khối theo bảng, khối sai dạng (mảng không phải mảng), độ phủ ngôn ngữ, object bản địa hoá chính thiếu `lang` |
| `reject_frame(...)` / `doc_reject_rows(df, ctx)` | Chuẩn hoá tài liệu lỗi theo schema `ctrl_cdc_reject` |

Luật phân loại (luật khớp đầu tiên quyết định):

| # | Điều kiện | Trạng thái | `reject_reason` |
|---|---|---|---|
| 1 | `source_name` / `source_id` NULL hoặc rỗng | REJECTED | `MISSING_ENTITY_KEY` |
| 2 | `source_name` ngoài tham số `sources` | IGNORED | `OUT_OF_SCOPE` |
| 3 | Payload NULL / rỗng / không phải JSON object | REJECTED | `INVALID_PAYLOAD` |
| 4 | `crawled_at` NULL | REJECTED | `MISSING_EVENT_ORDER` |


In [4]:
# =============================================================================
# A. ĐỌC, PHÂN LOẠI RAW, DỰNG TÀI LIỆU
# =============================================================================
def path_value(parsed: Column, path: str) -> Column:
    """Giá trị chuỗi tại đường dẫn: trường đầu lấy từ struct đã parse (parse_json_strings), phần sau bằng get_json_object.
    Vd path_value(F.col("_e"), "ext_attributes.primaryType") -> get_json_object(_e.ext_attributes, '$.primaryType')"""
    head, _, rest = path.partition(".")
    value = parsed.getField(head)
    return F.get_json_object(value, f"$.{rest}") if rest else value


def block_col(field: str) -> str:
    """Tên cột chứa trường cấp 1 của payload sau classify_docs. Vd "poi_address" -> "_b__poi_address"."""
    return f"_b__{field}"


def doc_value(path: str) -> Column:
    """Giá trị chuỗi tại đường dẫn tính từ gốc payload, đọc từ cột _b__<trường đầu> (sau classify_docs).
    Vd doc_value("extra_info.enrichment") -> get_json_object(_b__extra_info, '$.enrichment')"""
    head, _, rest = path.partition(".")
    value = F.col(block_col(head))
    return F.get_json_object(value, f"$.{rest}") if rest else value


def spec_paths(spec: TableSpec) -> list:
    """Mọi đường dẫn của các cột 1 bảng (cột khoá suy ra tách thành nhiều phần)."""
    return [p for c in spec.columns for p in c.paths]


def is_scoped(path: str) -> bool:
    """Đường dẫn có tiền tố đặc biệt (_doc, _root, _align, _lang, _pos, _langs) hoặc là '$'."""
    return path == "$" or path.split(".", 1)[0] in PATH_SCOPES


def top_fields(specs: list, need_langs: bool) -> list:
    """Trường cấp 1 của payload cần parse (1 lần cho mọi bảng): khối nguồn, đường dẫn _root.*, align_path,
    trường của bảng DOC '$', và 2 đường dẫn bản địa hoá nếu cần.
    Vd -> ["business_category", ..., "extra_info", "poi_address", "poi_review", ...]"""
    fields = set()
    for s in specs:
        if s.load_mode in ("DOC", "DOC_ARRAY") and s.src_object != "$":
            fields.add(s.src_object.split(".")[0])
        if s.align_path:
            fields.add(s.align_path.split(".")[0])
        for p in spec_paths(s):
            if p.startswith("_root."):
                fields.add(p[len("_root."):].split(".")[0])
            elif s.load_mode == "DOC" and s.src_object == "$" and not is_scoped(p):
                fields.add(p.split(".")[0])
    if need_langs:
        fields |= {lang_object_path.split(".")[0], lang_map_path.split(".")[0]}
    return sorted(fields)


def doc_columns(specs: list) -> list:
    """Cột raw cần đọc: cột cố định của nguồn + mọi `_doc.<cột>` trong registry."""
    cols = [RAW_PAYLOAD, RAW_EVENT_ID, *RAW_DOC_ID, RAW_LANG]
    for s in specs:
        for p in spec_paths(s):
            if p.startswith("_doc."):
                cols.append(p[len("_doc."):])
    return list(dict.fromkeys(cols))                              # bỏ trùng, giữ thứ tự


def uses_langs(specs: list) -> bool:
    """Có bảng LANG / LANG_ARRAY hoặc cột dùng _langs không (cần dựng object bản địa hoá)."""
    return any(s.load_mode in LANG_MODES or "_langs" in spec_paths(s) for s in specs)


def read_raw(ctx: RunContext, doc_cols: list) -> DataFrame:
    """Cột raw cần thiết + cột thứ tự (watermark_column, ép TIMESTAMP, tên _crawled_at) của phần raw cần xử lý.
    Mặc định VERSION: chỉ đọc file của các commit sau last_src_version. Cách đọc ghi vào ctx.read_mode / read_note."""
    return read_incremental(ctx, doc_cols, "_crawled_at")


def lang_entries() -> Column:
    """Object bản địa hoá của tài liệu: [(lang, obj JSON)], lang đã lower(trim); bỏ lang rỗng / obj NULL.
    Object chính (lang_object_path, lang ở trường lang_field) đứng trước map (lang_map_path) -> trùng lang thì object chính thắng
    (bước bỏ trùng ở classify_docs). Chưa lọc theo tham số langs. Object chính thiếu lang: bị bỏ (không đoán ngôn ngữ),
    đếm ở exit JSON langs.primary_without_lang."""
    primary = doc_value(lang_object_path)
    first = F.when(primary.isNotNull(), F.array(F.struct(
        F.lower(F.trim(F.get_json_object(primary, f"$.{lang_field}"))).alias("lang"), primary.alias("obj"))))
    siblings = F.from_json(doc_value(lang_map_path), MapType(StringType(), StringType()), JSON_OPTIONS)
    rest = F.transform(F.map_entries(siblings),
                       lambda e: F.struct(F.lower(F.trim(e["key"])).alias("lang"), e["value"].alias("obj")))
    empty = F.expr(f"CAST(array() AS {ENTRY_TYPE})")
    entries = F.concat(F.coalesce(first, empty), F.coalesce(rest, empty))
    return F.filter(entries, lambda e: e["lang"].isNotNull() & (e["lang"] != "") & e["obj"].isNotNull())


def classify_docs(raw: DataFrame, fields: list, need_langs: bool) -> DataFrame:
    """1 dòng / event: _status VALID | IGNORED | REJECTED, _reason, _reject_detail; payload parse 1 lần, mỗi trường cấp 1
    cần dùng thành 1 cột _b__<trường> (chỉ giữ cho VALID), payload gốc chỉ giữ cho REJECTED (ghi ctrl_cdc_reject).
    Kèm cột event theo hợp đồng lib §13 (_cdc_ts_ms = crawled_at ms, _cdc_lsn NULL, _raw_ingested_at = crawled_at,
    _is_delete 0, _op NULL) và ngôn ngữ: _langs_arr / _langs (mọi ngôn ngữ có trong tài liệu), _lang_entries (đã lọc
    theo langs, bỏ trùng lang), _primary_no_lang (object chính có nhưng thiếu lang)."""
    payload = F.col(RAW_PAYLOAD)
    df = raw.select(
        *[F.col(c).alias(f"_doc__{c}") for c in raw.columns if c not in ("_crawled_at", RAW_PAYLOAD)],
        F.col(RAW_EVENT_ID).cast("string").alias("_event_id"),
        F.col("_crawled_at"),
        payload.alias("_payload"),
        parse_json_strings(payload, fields).alias("_p"),
    )

    def blank(c: Column) -> Column:
        return c.isNull() | (F.trim(c.cast("string")) == "")

    scope = [s.lower() for s in split_csv(sources)]
    out_of_scope = (~F.lower(F.trim(F.col(f"_doc__{RAW_DOC_ID[0]}"))).isin(scope)) if scope else F.lit(False)
    rules = [   # (điều kiện, lý do, chi tiết) — thứ tự ưu tiên từ trên xuống
        *[(blank(F.col(f"_doc__{c}")), "MISSING_ENTITY_KEY", F.lit(f"thiếu {c}")) for c in RAW_DOC_ID],
        (out_of_scope, "OUT_OF_SCOPE", None),
        (blank(F.col("_payload")), "INVALID_PAYLOAD", F.lit(f"thiếu {RAW_PAYLOAD}")),
        (is_json_corrupt(F.col("_p"), F.col("_payload")), "INVALID_PAYLOAD",
         F.lit(f"{RAW_PAYLOAD} không phải JSON object hợp lệ")),
        (F.col("_crawled_at").isNull(), "MISSING_EVENT_ORDER", F.lit("thiếu cột thứ tự (watermark_column)")),
    ]
    reason, detail = F.lit(None).cast("string"), F.lit(None).cast("string")
    for cond, r, d in reversed(rules):
        reason = F.when(cond, F.lit(r)).otherwise(reason)
        detail = F.when(cond, d if d is not None else F.lit(None).cast("string")).otherwise(detail)

    df = (
        df.withColumn("_reason", reason)
          .withColumn("_reject_detail", detail)
          .withColumn("_status", F.when(F.col("_reason").isNull(), "VALID")
                                  .when(F.col("_reason").isin(*IGNORE_REASONS), "IGNORED")
                                  .otherwise("REJECTED"))
          .withColumn("_cdc_ts_ms", F.expr("unix_millis(_crawled_at)"))
          .withColumn("_cdc_lsn", F.lit(None).cast("bigint"))
          .withColumn("_raw_ingested_at", F.col("_crawled_at"))
          .withColumn("_is_delete", F.lit(0))
          .withColumn("_op", F.lit(None).cast("string"))
    )
    # Tách struct _p thành cột _b__<trường> (chỉ VALID): cache dạng cột -> mỗi bảng chỉ đọc khối mình cần
    valid = F.col("_status") == "VALID"
    df = df.select(*[c for c in df.columns if c != "_p"],
                   *[F.when(valid, F.col("_p").getField(f)).alias(block_col(f)) for f in fields])
    if need_langs:
        allowed = [x.lower() for x in split_csv(langs)]
        primary = doc_value(lang_object_path)
        df = (df.withColumn("_le_all", F.when(valid, lang_entries()))
                .withColumn("_primary_no_lang", valid & primary.isNotNull()
                            & F.coalesce(F.trim(F.get_json_object(primary, f"$.{lang_field}")), F.lit("")).eqNullSafe("")))
        df = (
            df.withColumn("_langs_arr", F.array_sort(F.array_distinct(F.filter(
                    F.concat(F.array(F.lower(F.trim(F.col(f"_doc__{RAW_LANG}").cast("string")))),
                             F.coalesce(F.transform("_le_all", lambda e: e["lang"]), F.expr("CAST(array() AS array<string>)"))),
                    lambda x: x.isNotNull() & (x != "")))))
              .withColumn("_langs", F.to_json("_langs_arr"))
              # bỏ trùng lang (giữ lần xuất hiện đầu = object chính), rồi lọc theo tham số langs
              .withColumn("_lang_entries", F.expr(
                  "filter(_le_all, (e, i) -> i = array_position(transform(_le_all, x -> x.lang), e.lang) - 1)"))
              .withColumn("_lang_entries", F.filter("_lang_entries", lambda e: e["lang"].isin(allowed))
                          if allowed else F.col("_lang_entries"))
              .drop("_le_all")
        )
    # Chỉ giữ thứ cần cho bước sau -> cache nhẹ: khối _b__* cho VALID (ở trên), payload gốc cho REJECTED
    return df.withColumn("_payload", F.when(F.col("_status") == "REJECTED", F.col("_payload")))


def presence(spec: TableSpec) -> Column:
    """Tài liệu VALID có khối của bảng không (NO_DATA khi không tài liệu nào có).
    DOC '$': luôn có ; DOC: khối khác NULL ; DOC_ARRAY: mảng có ≥ 1 phần tử ; LANG*: có ≥ 1 object bản địa hoá đã lọc."""
    if spec.load_mode in LANG_MODES:
        return F.size("_lang_entries") > 0
    if spec.src_object == "$":
        return F.lit(True)
    block = doc_value(spec.src_object)
    if spec.load_mode == "DOC_ARRAY":
        return F.size(F.from_json(block, ArrayType(StringType()), JSON_OPTIONS)) > 0
    return block.isNotNull()


def bad_shape(spec: TableSpec) -> Column:
    """Khối DOC_ARRAY có giá trị nhưng không phải mảng JSON (vd object): bảng không nhận được dòng nào từ tài liệu đó.
    Không reject (tài liệu vẫn đúng với bảng khác), đếm ở exit JSON bad_shape + WARN."""
    block = doc_value(spec.src_object)
    return block.isNotNull() & F.from_json(block, ArrayType(StringType()), JSON_OPTIONS).isNull()


def profile_docs(classified: DataFrame, specs: list, need_langs: bool) -> tuple:
    """Đếm sau khi đọc raw (đồng thời nạp cache):
    - Profile: read/valid/ignored/rejected, ignored_detail {lý do: {source_name: n}}, watermark_to = max(crawled_at)
    - {trg_tbl: số tài liệu có khối} (0 -> NO_DATA)
    - độ phủ ngôn ngữ: {"missing": {lang: số tài liệu thiếu}, "not_allowed": {lang: số tài liệu có lang ngoài langs},
      "primary_without_lang": số tài liệu có object bản địa hoá chính nhưng thiếu lang (bị bỏ)}
    - {trg_tbl: số tài liệu có khối DOC_ARRAY sai dạng}
    Output: (Profile, dict, dict, dict)"""
    src_col = F.col(f"_doc__{RAW_DOC_ID[0]}")
    rows = (classified.groupBy("_status", "_reason", src_col.alias("_src"))
            .agg(F.count(F.lit(1)).alias("n"), F.max("_crawled_at").alias("max_ts")).collect())
    p = Profile()
    for r in rows:
        n = r["n"]
        p.read_rows += n
        if r["max_ts"] is not None and (p.watermark_to is None or r["max_ts"] > p.watermark_to):
            p.watermark_to = r["max_ts"]
        if r["_status"] == "VALID":
            p.valid_rows += n
        elif r["_status"] == "IGNORED":
            p.ignored_rows += n
            bucket = p.ignored_detail.setdefault(r["_reason"], {})
            bucket[r["_src"] or "(null)"] = bucket.get(r["_src"] or "(null)", 0) + n
        else:
            p.rejected_rows += n

    valid = classified.where("_status = 'VALID'")
    allowed = [x.lower() for x in split_csv(langs)]
    aggs = [F.sum(F.when(presence(s), 1).otherwise(0)).alias(f"t{i}") for i, s in enumerate(specs)]
    aggs += [F.sum(F.when(bad_shape(s), 1).otherwise(0)).alias(f"b{i}")
             for i, s in enumerate(specs) if s.load_mode == "DOC_ARRAY"]
    if need_langs:
        aggs += [F.sum(F.when(~F.array_contains("_langs_arr", l), 1).otherwise(0)).alias(f"m_{l}") for l in allowed]
        aggs.append(F.sum(F.when(F.col("_primary_no_lang"), 1).otherwise(0)).alias("no_lang"))
    agg = valid.agg(*aggs).first()
    counts = {s.trg_tbl: int(agg[f"t{i}"] or 0) for i, s in enumerate(specs)}
    shapes = {s.trg_tbl: int(agg[f"b{i}"] or 0) for i, s in enumerate(specs)
              if s.load_mode == "DOC_ARRAY" and agg[f"b{i}"]}
    lang_stats = {}
    if need_langs:
        lang_stats["missing"] = {l: int(agg[f"m_{l}"] or 0) for l in allowed if agg[f"m_{l}"]}
        if agg["no_lang"]:
            lang_stats["primary_without_lang"] = int(agg["no_lang"])
        if allowed:
            extra = (valid.select(F.explode(F.array_except("_langs_arr", F.array(*[F.lit(l) for l in allowed])))
                                  .alias("lang")).groupBy("lang").count().collect())
            lang_stats["not_allowed"] = {r["lang"]: int(r["count"]) for r in extra}
    return p, counts, lang_stats, shapes


def reject_frame(df: DataFrame, ctx: RunContext, event_hash: Column, src_object: Column, trg_schema: Column,
                 trg_tbl: Column, entity_key: Column, reason: Column, detail: Column, after: Column,
                 source_payload: Column) -> DataFrame:
    """Dòng ctrl_cdc_reject của nguồn 3rd-party (map cột: src_db = source_name, cdc_op NULL, cdc_ts_ms = crawled_at ms,
    raw_ingested_at = crawled_at, before NULL, after = payload / phần tử lỗi, source = JSON định danh event)."""
    now = F.current_timestamp()
    return df.select(
        event_hash.alias("event_hash"),
        F.lit(ctx.pl_name).alias("pl_name"),
        F.lit(ctx.src_schema).alias("src_schema"),
        F.lit(ctx.src_tbl).alias("src_tbl"),
        src_object.cast("string").alias("src_object"),
        F.col(f"_doc__{RAW_DOC_ID[0]}").cast("string").alias("src_db"),
        trg_schema.cast("string").alias("trg_schema"),
        trg_tbl.cast("string").alias("trg_tbl"),
        F.lit(None).cast("string").alias("cdc_op"),
        entity_key.cast("string").alias("entity_key"),
        F.col("_cdc_ts_ms").alias("cdc_ts_ms"),
        F.col("_cdc_lsn").alias("cdc_lsn"),
        F.col("_raw_ingested_at").alias("raw_ingested_at"),
        reason.alias("reject_reason"),
        detail.alias("reject_detail"),
        F.lit(None).cast("string").alias("before_payload"),
        after.cast("string").alias("after_payload"),
        source_payload.alias("source_payload"),
        F.lit(ctx.run_id).alias("first_run_id"),
        F.lit(ctx.run_id).alias("last_run_id"),
        F.lit(1).alias("reject_count"),
        now.alias("first_rejected_at"),
        now.alias("last_rejected_at"),
        F.lit(False).alias("is_resolved"),
        F.lit(None).cast("timestamp").alias("resolved_at"),
    )


def doc_source_payload(extra: Optional[dict] = None) -> Column:
    """JSON định danh event cho cột source_payload: event_id, các cột RAW_DOC_ID, RAW_LANG, crawled_at (+ extra)."""
    fields = [F.col("_event_id").alias(RAW_EVENT_ID),
              *[F.col(f"_doc__{c}").alias(c) for c in (*RAW_DOC_ID, RAW_LANG)],
              F.col("_crawled_at").alias("crawled_at")]
    fields += [v.alias(k) for k, v in (extra or {}).items()]
    return F.to_json(F.struct(*fields))


def doc_reject_rows(df: DataFrame, ctx: RunContext) -> DataFrame:
    """Tài liệu lỗi phân loại -> ctrl_cdc_reject. event_hash theo event (event_id, định danh, crawled_at, payload):
    cùng event đọc lại (FULL) -> reject_count + 1, không sinh dòng trùng."""
    ident = [F.coalesce(F.col(c).cast("string"), F.lit(""))
             for c in ("_event_id", *[f"_doc__{x}" for x in RAW_DOC_ID], "_crawled_at", "_payload")]
    doc_id = [F.col(f"_doc__{c}").cast("string") for c in RAW_DOC_ID]
    return reject_frame(
        df, ctx,
        event_hash=F.sha2(F.concat_ws("\u0001", *ident), 256),
        src_object=F.lit(None), trg_schema=F.lit(None), trg_tbl=F.lit(None),
        entity_key=F.when(F.coalesce(*doc_id).isNotNull(), F.concat_ws("|", *doc_id)),
        reason=F.col("_reason"), detail=F.col("_reject_detail"),
        after=F.col("_payload"), source_payload=doc_source_payload(),
    )

StatementMeta(, 611ec303-fa15-471c-a8d4-04a943aaf370, 19, Finished, Available, Finished, False)

## B. Xử lý 1 bảng: tách dòng → chuyển kiểu, khoá → lọc theo state → bản mới nhất → MERGE

| Hàm | Mục đích |
|---|---|
| `expand_rows(spec, docs)` | Tách dòng theo `load_mode`: `_elem` (phần tử/khối JSON), `_pos`, `_lang`, `_align` (mảng align parse 1 lần / tài liệu, trước khi nổ) |
| `part_value(path)` | Chuỗi gốc của 1 đường dẫn theo tiền tố (`$`, `_doc.`, `_root.`, `_align.`, `_lang`, `_pos`, `_langs`, trường của phần tử) |
| `build_table_rows(spec, docs, state_df)` | 1 dòng/phần tử: cột đã chuyển kiểu + cờ `_parse_failed` (chỉ khi bảng cần trường của phần tử), `_key_null`, `_is_stale`, `_applicable`, `_cn__<cột>` |
| `table_stats(spec, rows)` | 1 job đếm mọi số liệu của bảng |
| `resolve_latest(spec, rows)` | 1 dòng / khoá bảng: tài liệu mới nhất `(crawled_at, event_id)`, rồi `dedup_order`, rồi `_pos`; kèm `_first_seen_at` / `_last_seen_at` |
| `build_merge_sql(spec, target, view)` | MERGE: khớp + `(_crawled_at, _event_id)` mới ≥ đang lưu → cập nhật; không khớp → insert. Không có nhánh xoá |
| `merge_into_target(spec, latest, ctx)` | Chạy MERGE qua `run_merge` của thư viện |
| `doc_state_latest(rows)` | 1 dòng / `poi_id` (tài liệu mới nhất đã áp dụng) để ghi `ctrl_cdc_state` bằng `state_rows` của thư viện |
| `element_reject_rows(spec, rows, ctx)` | Phần tử thiếu khoá / không phải JSON object → `ctrl_cdc_reject` (event_hash theo nội dung phần tử: crawl lại không sinh dòng trùng) |
| `_run_table` / `process_table` | Các bước của 1 bảng; `cast_null_policy = FAIL` → `CastNullError` trước MERGE; retry + log: `process_with_retry` (lib §14) |

State (tham số `state_key`):

| state_key | entity_key | Tài liệu cũ đến trễ (crawl trước bản đã áp dụng) | Số dòng state |
|---|---|---|---|
| `doc` (mặc định) | `poi_id` cho mọi bảng | Bỏ qua cả tài liệu ở mọi bảng đã có bản mới hơn — phần tử chỉ có ở bản cũ đó (vd 1 review) không vào bảng con | 14 × số địa điểm |
| `row` | Khoá bảng (`review_id`, `poi_id\|locale`...) | Phần tử chưa có state vẫn vào; phần tử đã có bản mới hơn bị bỏ qua | ~3 lần |

Kết quả bảng đích như nhau khi tài liệu đến đúng thứ tự (MERGE luôn chặn ghi lùi theo `(_crawled_at, _event_id)`).
Chọn `row` nếu Q6 (POI_3P_EXTRACT_REDESIGN §6.1) cho thấy có tài liệu đến trễ. Đổi `state_key` không cần reset state.
Bảng con (DOC_ARRAY, LANG*) chỉ upsert: phần tử không còn trong tài liệu mới vẫn giữ, `_last_seen_at` dừng ở lần cuối thấy.


In [5]:
# =============================================================================
# B. XỬ LÝ 1 BẢNG
# =============================================================================
def expand_rows(spec: TableSpec, docs: DataFrame) -> DataFrame:
    """Tài liệu VALID -> dòng của bảng theo load_mode. Giữ cột event + _doc__* + _langs + cột khối _b__* bảng cần
    (bảng DOC '$' và đường dẫn _root.*). Cột thêm: _elem (chuỗi JSON phần tử/khối; NULL với DOC '$'), _pos (vị trí, *_ARRAY),
    _lang (LANG*), _align (phần tử cùng vị trí trong align_path, chỉ khi 2 mảng cùng số phần tử).
    Mảng NULL / rỗng: posexplode không sinh dòng (không lọc trước để from_json không bị tính lại)."""
    paths = spec_paths(spec)
    root_doc = spec.load_mode == "DOC" and spec.src_object == "$"
    blocks = {p[len("_root."):].split(".")[0] for p in paths if p.startswith("_root.")}
    if root_doc:
        blocks |= {p.split(".")[0] for p in paths if not is_scoped(p)}
    meta = ["_event_id", "_crawled_at", "_cdc_ts_ms", "_cdc_lsn", "_raw_ingested_at", "_is_delete", "_op",
            *[c for c in docs.columns if c.startswith("_doc__")],
            *(["_langs"] if "_langs" in docs.columns else []), *[block_col(b) for b in sorted(blocks)]]
    arr_t = ArrayType(StringType())

    if spec.load_mode == "DOC":
        if root_doc:
            return docs.select(*meta, F.lit(None).cast("string").alias("_elem"))
        block = doc_value(spec.src_object)
        return docs.where(block.isNotNull()).select(*meta, block.alias("_elem"))

    # Mảng align tính 1 lần / tài liệu, trước khi nổ
    align = [F.from_json(doc_value(spec.align_path), arr_t, JSON_OPTIONS).alias("_al")] if spec.align_path else []
    keep_al = ["_al"] if spec.align_path else []
    if spec.load_mode == "DOC_ARRAY":
        rows = docs.select(*meta, *align, F.from_json(doc_value(spec.src_object), arr_t, JSON_OPTIONS).alias("_arr"))
    else:                                                        # LANG, LANG_ARRAY: 1 dòng / object bản địa hoá
        rows = (docs.select(*meta, *align, F.explode("_lang_entries").alias("_le"))
                    .select(*meta, *keep_al, F.col("_le.lang").alias("_lang"), F.col("_le.obj").alias("_lobj")))
        meta = meta + ["_lang"]
        inner = F.col("_lobj") if spec.src_object == "$" else F.get_json_object("_lobj", f"$.{spec.src_object}")
        if spec.load_mode == "LANG":
            return rows.where(inner.isNotNull()).select(*meta, inner.alias("_elem"))
        rows = rows.select(*meta, *keep_al, F.from_json(inner, arr_t, JSON_OPTIONS).alias("_arr"))

    rows = rows.select(*meta, *keep_al, F.size("_arr").alias("_n"), F.posexplode("_arr").alias("_pos", "_elem"))
    if spec.align_path:
        rows = rows.withColumn("_align", F.when(F.size("_al") == F.col("_n"), F.element_at("_al", F.col("_pos") + 1)))
    return rows.drop("_al", "_n")


def part_value(path: str) -> Column:
    """Chuỗi gốc của 1 đường dẫn (bảng json_path ở lib §1). Trường của phần tử lấy từ struct _e đã parse."""
    if path == "$":
        return F.col("_elem")
    if path == "_lang":
        return F.col("_lang")
    if path == "_pos":
        return F.col("_pos").cast("string")
    if path == "_langs":
        return F.col("_langs")
    head, _, rest = path.partition(".")
    if head == "_doc":
        return F.col(f"_doc__{rest}").cast("string")
    if head == "_root":
        return doc_value(rest)
    if head == "_align":
        return F.get_json_object(F.col("_align"), f"$.{rest}")
    return path_value(F.col("_e"), path)


def build_table_rows(spec: TableSpec, docs: DataFrame, state_df: Optional[DataFrame]) -> DataFrame:
    """Dòng của 1 bảng với cột đã chuyển kiểu và các cờ:
    _parse_failed (bảng cần trường của phần tử nhưng phần tử không phải JSON object), _key_null (thiếu khoá bảng hoặc
    doc_key_column -> reject), _is_stale (tài liệu cũ hơn hoặc bằng state của _entity_key: poi_id khi state_key = "doc",
    khoá bảng khi "row"), _applicable, _cn__<cột> (giá trị có ở raw nhưng thành NULL do chuyển kiểu).
    _elem/_align chỉ giữ cho dòng reject."""
    rows = expand_rows(spec, docs)
    root_doc = spec.load_mode == "DOC" and spec.src_object == "$"
    elem_fields = sorted({p.split(".")[0] for p in spec_paths(spec) if not is_scoped(p)})
    if root_doc:                                                 # trường gốc payload: đã là cột _b__*
        rows = (rows.withColumn("_e", F.struct(*[F.col(block_col(f)).alias(f) for f in elem_fields]))
                    .withColumn("_parse_failed", F.lit(False)))
    elif elem_fields:                                            # cần trường của phần tử -> phần tử phải là JSON object
        rows = (rows.withColumn("_e", parse_json_strings(F.col("_elem"), elem_fields))
                    .withColumn("_parse_failed", is_json_corrupt(F.col("_e"), F.col("_elem"))))
    else:                                                        # chỉ lấy cả khối ($) / tiền tố: khối dạng gì cũng nhận
        rows = rows.withColumn("_parse_failed", F.lit(False))

    raw_cols = []
    for c in spec.columns:
        if c.is_derived:
            raw_cols += [part_value(p).alias(n) for p, n in zip(c.paths, c.part_cols)]
        else:
            raw_cols.append(part_value(c.json_path).alias(c.raw_col))
    meta = [x for x in rows.columns if x != "_e" and not x.startswith("_b__")]   # khối gốc không mang theo cache
    rows = rows.select(*meta, *raw_cols)

    def typed(c: ColumnSpec) -> Column:
        sql = derived_sql(c.part_cols, c.convert_rule) if c.is_derived else convert_sql(c.raw_col, c.data_type, c.convert_rule)
        return F.expr(sql).alias(c.name)

    rows = rows.select(*meta, *[F.col(c.raw_col) for c in spec.columns if not c.is_derived],
                       *[typed(c) for c in spec.columns])
    rows = rows.select(*meta, *[c.name for c in spec.columns],
                       *[(F.col(c.raw_col).isNotNull() & F.col(c.name).isNull()).alias(f"_cn__{c.name}")
                         for c in spec.columns if c.can_cast_null])

    keys = spec.key_cols
    needed = list(dict.fromkeys(keys + [doc_key_column]))
    key_null = F.col("_parse_failed") | reduce(
        lambda a, b: a | b, [F.col(k).isNull() | (F.trim(F.col(k).cast("string")) == "") for k in needed])
    row_key = F.concat_ws("|", *[F.col(k).cast("string") for k in keys])
    entity = F.col(doc_key_column).cast("string") if state_key == "doc" else row_key    # tham số state_key
    rows = (rows.withColumn("_key_null", key_null)
                .withColumn("_entity_key", F.when(~F.col("_key_null"), entity))
                .withColumn("_row_key", F.when(~F.col("_key_null"), row_key)))

    if state_df is None:                                         # full_reload: không dùng state
        is_stale = F.lit(False)
    else:
        st = (state_df.where((F.col("trg_schema") == spec.trg_schema) & (F.col("trg_tbl") == spec.trg_tbl))
                      .select("_st_key", "_st_order"))
        rows = rows.join(st, F.col("_entity_key") == F.col("_st_key"), "left")
        is_stale = F.col("_st_key").isNotNull() & ~(event_order() > F.col("_st_order"))
    rows = (rows.withColumn("_is_stale", ~F.col("_key_null") & F.coalesce(is_stale, F.lit(False)))
                .withColumn("_applicable", ~F.col("_key_null") & ~F.col("_is_stale")))
    for c in ("_elem", "_align"):                                # phần tử gốc chỉ giữ cho dòng reject -> cache nhẹ
        if c in rows.columns:
            rows = rows.withColumn(c, F.when(F.col("_key_null"), F.col(c)))
    return rows.drop("_st_key", "_st_order")


def table_stats(spec: TableSpec, rows: DataFrame) -> dict:
    """1 job cho toàn bộ số đếm của bảng. Với nguồn 3rd-party: input_rows = số dòng sau khi tách (phần tử),
    entity_rows = số khoá bảng đưa vào MERGE, upsert_event_rows = applicable_rows, delete_event_rows = 0."""
    applicable = F.col("_applicable")

    def as_int(cond: Column) -> Column:
        return F.sum(F.when(cond, 1).otherwise(0))

    aggs = [
        F.count(F.lit(1)).alias("input_rows"),
        as_int(F.col("_key_null")).alias("rejected_rows"),
        as_int(F.col("_is_stale")).alias("stale_rows"),
        as_int(applicable).alias("applicable_rows"),
        F.countDistinct(F.when(applicable, F.col("_row_key"))).alias("entity_rows"),
    ]
    aggs += [as_int(applicable & F.col(f"_cn__{c.name}")).alias(f"_cn__{c.name}")
             for c in spec.columns if c.can_cast_null]
    row = rows.agg(*aggs).first().asDict()
    out = {k: int(v or 0) for k, v in row.items()}
    out["upsert_event_rows"], out["delete_event_rows"] = out["applicable_rows"], 0
    return out


def resolve_latest(spec: TableSpec, rows: DataFrame) -> DataFrame:
    """Dòng được áp dụng -> 1 dòng / khoá bảng. Thứ tự chọn: tài liệu mới nhất (crawled_at, event_id), rồi dedup_order
    của bảng (NULL xếp cuối), rồi vị trí trong mảng. _first_seen_at / _last_seen_at = min / max crawled_at của khoá trong lần chạy."""
    order = [F.col("_cdc_ts_ms").desc(), F.col("_raw_ingested_at").desc(), F.col("_event_id").desc()]
    order += [F.col(n).desc_nulls_last() if desc else F.col(n).asc_nulls_last() for n, desc in spec.dedup_order]
    if "_pos" in rows.columns:
        order.append(F.col("_pos").asc_nulls_last())
    by_key = Window.partitionBy(*spec.key_cols)
    return (rows.withColumn("_first_seen_at", F.min("_crawled_at").over(by_key))
                .withColumn("_last_seen_at", F.max("_crawled_at").over(by_key))
                .withColumn("_rn", F.row_number().over(by_key.orderBy(*order)))
                .where("_rn = 1").drop("_rn"))


def build_merge_sql(spec: TableSpec, target: str, view: str) -> str:
    """Câu MERGE: khớp và (_crawled_at, _event_id) nguồn ≥ đích -> cập nhật cột + _crawled_at/_event_id, _first_seen_at =
    nhỏ nhất, _last_seen_at = lớn nhất; không khớp -> insert. Không có nhánh xoá (nguồn không có tín hiệu xoá).
    Điều kiện thứ tự chặn ghi lùi ở mức dòng (vd full_reload, run cũ commit muộn); trùng crawled_at thì event_id lớn hơn
    thắng — cùng quy tắc với resolve_latest."""
    on = " AND ".join(f"t.`{k}` = s.`{k}`" for k in spec.key_cols)
    updates = [f"t.`{c.name}` = s.`{c.name}`" for c in spec.columns if not c.is_key]
    updates += ["t.`_crawled_at` = s.`_crawled_at`", "t.`_event_id` = s.`_event_id`",
                "t.`_first_seen_at` = least(t.`_first_seen_at`, s.`_first_seen_at`)",
                "t.`_last_seen_at` = greatest(t.`_last_seen_at`, s.`_last_seen_at`)",
                "t.`_ingested_at` = current_timestamp()"]
    insert_cols = [f"`{c.name}`" for c in spec.columns] + [f"`{n}`" for n, _ in SNAPSHOT_TECH_COLUMNS]
    insert_vals = [f"s.`{c.name}`" for c in spec.columns] + [
        "s.`_crawled_at`", "s.`_event_id`", "s.`_first_seen_at`", "s.`_last_seen_at`", "current_timestamp()"]
    sep = ",\n        "
    return (
        f"MERGE INTO {target} AS t\n"
        f"USING {view} AS s\n"
        f"ON {on}\n"
        f"WHEN MATCHED AND (s.`_crawled_at` > coalesce(t.`_crawled_at`, TIMESTAMP'{MIN_TS:%Y-%m-%d %H:%M:%S}')\n"
        f"                  OR (s.`_crawled_at` = t.`_crawled_at` AND s.`_event_id` >= coalesce(t.`_event_id`, ''))) THEN UPDATE SET\n"
        f"        {sep.join(updates)}\n"
        f"WHEN NOT MATCHED THEN INSERT\n"
        f"        ({', '.join(insert_cols)})\n"
        f"        VALUES ({', '.join(insert_vals)})"
    )


def merge_into_target(spec: TableSpec, latest: DataFrame, ctx: RunContext) -> tuple:
    """MERGE dòng mới nhất mỗi khoá vào bảng đích -> (inserted, updated). Temp view tên riêng theo exec_id + bảng."""
    source = latest.select(*[c.name for c in spec.columns], "_crawled_at", "_event_id", "_first_seen_at", "_last_seen_at")
    return run_merge(source, spec.fq_name, lambda target, view: build_merge_sql(spec, target, view),
                     f"_v_{ctx.exec_id[:8]}_{spec.trg_tbl}")


def doc_state_latest(rows: DataFrame) -> DataFrame:
    """Dòng được áp dụng -> 1 dòng / _entity_key (poi_id) theo tài liệu mới nhất, đủ cột cho state_rows (lib §13)."""
    w = Window.partitionBy("_entity_key").orderBy(F.col("_cdc_ts_ms").desc(), F.col("_raw_ingested_at").desc(),
                                                  F.col("_event_id").desc())
    return (rows.select("_entity_key", "_op", "_cdc_ts_ms", "_cdc_lsn", "_raw_ingested_at", "_event_id")
                .withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn"))


def element_reject_rows(spec: TableSpec, rows: DataFrame, ctx: RunContext) -> DataFrame:
    """Phần tử lỗi của 1 bảng -> ctrl_cdc_reject. event_hash = sha256(bảng, định danh địa điểm, lang, phần tử, phần tử align):
    cùng phần tử lỗi ở các lần crawl sau -> reject_count + 1 thay vì thêm dòng."""
    needed = list(dict.fromkeys(spec.key_cols + [doc_key_column]))
    missing = F.concat_ws(",", *[F.when(F.col(k).isNull() | (F.trim(F.col(k).cast("string")) == ""), F.lit(k))
                                 for k in needed])
    note = f" (align_path {spec.align_path} khác số phần tử?)" if spec.align_path else ""
    opt = lambda c: F.coalesce(F.col(c).cast("string"), F.lit("")) if c in rows.columns else F.lit("")
    ident = [F.lit(spec.fq_name), *[opt(f"_doc__{c}") for c in RAW_DOC_ID], opt("_lang"), opt("_elem"), opt("_align")]
    extra = {k[1:]: F.col(k) for k in ("_lang", "_pos") if k in rows.columns}     # source_payload thêm lang, pos
    return reject_frame(
        rows, ctx,
        event_hash=F.sha2(F.concat_ws("\u0001", *ident), 256),
        src_object=F.lit(spec.src_object), trg_schema=F.lit(spec.trg_schema), trg_tbl=F.lit(spec.trg_tbl),
        entity_key=F.col(doc_key_column),
        reason=F.when(F.col("_parse_failed"), F.lit("INVALID_PAYLOAD")).otherwise(F.lit("MISSING_ENTITY_KEY")),
        detail=F.when(F.col("_parse_failed"), F.lit(f"phần tử của {spec.src_object} không phải JSON object"))
                .otherwise(F.concat(F.lit("khoá NULL/rỗng: "), missing, F.lit(note))),
        after=F.col("_elem") if "_elem" in rows.columns else F.lit(None),
        source_payload=doc_source_payload(extra),
    )


def _run_table(res: TableResult, docs: DataFrame, state_df: Optional[DataFrame], ctx: RunContext) -> None:
    """Các bước của 1 bảng; ghi kết quả vào res. Lỗi -> raise cho process_with_retry xử lý."""
    spec = res.spec
    note = ensure_target(spec, ctx.dry_run)                      # cột kỹ thuật theo spec: SNAPSHOT_TECH_COLUMNS
    if note:
        log(note, wave=res.wave, table=spec.trg_tbl)

    rows = build_table_rows(spec, docs, state_df).persist(StorageLevel.MEMORY_AND_DISK)
    res.cached.append(rows)
    stats = table_stats(spec, rows)
    for k in ("input_rows", "rejected_rows", "stale_rows", "applicable_rows",
              "entity_rows", "upsert_event_rows", "delete_event_rows"):
        setattr(res, k, stats[k])
    res.cast_null_detail = {k[len("_cn__"):]: v for k, v in stats.items() if k.startswith("_cn__") and v}
    res.cast_null_rows = sum(res.cast_null_detail.values())
    if res.cast_null_detail:
        if ctx.cast_null_policy == "FAIL" and not ctx.dry_run:           # dry_run: chỉ cảnh báo để xem đủ mọi bảng
            raise CastNullError(
                f"{res.cast_null_rows} giá trị thành NULL khi chuyển kiểu {res.cast_null_detail}. Sửa data_type/"
                f"convert_rule trong ctrl_cfg_schema_registry rồi chạy lại bình thường; hoặc chấp nhận NULL: chạy "
                f"cast_null_policy = WARN")
        log(f"giá trị thành NULL khi chuyển kiểu: {res.cast_null_detail}", "WARN", res.wave, spec.trg_tbl)

    if res.rejected_rows:
        res.reject_df = element_reject_rows(spec, rows.where("_key_null"), ctx)
    if res.applicable_rows == 0 or ctx.dry_run:
        return

    applicable = rows.where("_applicable")
    latest = resolve_latest(spec, applicable).persist(StorageLevel.MEMORY_AND_DISK)
    res.cached.append(latest)
    res.inserted_rows, res.updated_rows = merge_into_target(spec, latest, ctx)
    res.state_df = state_rows(spec, doc_state_latest(applicable), ctx)


def process_table(spec: TableSpec, docs: DataFrame, state_df: Optional[DataFrame],
                  ctx: RunContext, wave: int) -> TableResult:
    """Chạy trong luồng của wave; retry lỗi tạm thời, không bao giờ raise (process_with_retry của lib §14)."""
    return process_with_retry(spec, wave, ctx, lambda res: _run_table(res, docs, state_df, ctx))

StatementMeta(, 611ec303-fa15-471c-a8d4-04a943aaf370, 20, Finished, Available, Finished, False)

## C. Commit

| Hàm | Mục đích |
|---|---|
| `commit(classified, profile, results, ctx)` | Thứ tự: state (`commit_state`) → reject (tài liệu lỗi + phần tử lỗi của bảng SUCCESS, `merge_rejects`) → table log. Chỉ luồng chính ghi bảng control |

In [6]:
# =============================================================================
# C. COMMIT
# =============================================================================
def commit(classified: DataFrame, profile: Profile, results: list, ctx: RunContext) -> None:
    """Thứ tự: state -> reject -> table log. Chết giữa chừng: lần sau đọc lại, state lọc tài liệu đã áp dụng."""
    commit_state(results, ctx.src_tbl)       # src_tbl: MERGE chỉ đọc partition của nguồn này (không xung đột với partner)
    parts = [r.reject_df for r in results if r.status == "SUCCESS" and r.reject_df is not None]
    if profile.rejected_rows:
        parts.append(doc_reject_rows(classified.where("_status = 'REJECTED'"), ctx))
    merge_rejects(parts, f"tài liệu lỗi={profile.rejected_rows}, "
                         f"phần tử lỗi={sum(r.rejected_rows for r in results if r.status == 'SUCCESS')}", ctx.src_tbl)
    append_rows(T_LOG_TABLE_RUN, [table_log_row(r, ctx) for r in results])

StatementMeta(, 611ec303-fa15-471c-a8d4-04a943aaf370, 21, Finished, Available, Finished, False)

## Main

`build_context()` đọc tham số thành `RunContext`; `check_config()` kiểm tra phần riêng của nguồn (cột tài liệu, cột raw);
`run_extract()` nạp cấu hình + watermark rồi gọi `run_controlled` (lib §14: khoá, run log, watermark, nhả khoá);
phần riêng của nguồn nằm ở `process_run()`.

In [7]:
# =============================================================================
# MAIN
# =============================================================================
def build_context() -> RunContext:
    """Tham số notebook -> RunContext (kiểm tra tham số riêng của nguồn). run_mode: DRY_RUN > FULL_RELOAD > INCREMENTAL."""
    errors = []
    if not is_fq_name(f"{src_schema}.{src_tbl}"):
        errors.append(f"src_schema/src_tbl không hợp lệ: {src_schema}.{src_tbl}")
    policy = str(cast_null_policy or "").strip().upper()
    if policy not in CAST_NULL_POLICIES:
        errors.append(f"cast_null_policy phải là {' | '.join(CAST_NULL_POLICIES)}, đang là {cast_null_policy!r}")
    for name, value in (("lang_object_path", lang_object_path), ("lang_map_path", lang_map_path)):
        if not JSON_PATH_RE.fullmatch(str(value or "")):
            errors.append(f"{name} không hợp lệ: {value!r}")
    for name, value in (("lang_field", lang_field), ("doc_key_column", doc_key_column)):
        if not is_identifier(str(value or "")):
            errors.append(f"{name} không hợp lệ: {value!r}")
    if state_key not in ("doc", "row"):
        errors.append(f"state_key phải là doc | row, đang là {state_key!r}")
    if errors:
        raise ConfigError("Tham số sai:\n  - " + "\n  - ".join(errors))
    exec_id = str(uuid.uuid4())
    filters = split_csv(table_filter)
    params = {
        "pl_name": pl_name, "run_id": run_id, "src_schema": src_schema, "src_tbl": src_tbl,
        "max_parallel": int(max_parallel), "max_retries": int(max_retries), "table_filter": filters,
        "full_reload": to_bool(full_reload), "dry_run": to_bool(dry_run), "stop_on_failure": to_bool(stop_on_failure),
        "running_timeout_minutes": int(running_timeout_minutes), "allow_full_scan": to_bool(allow_full_scan),
        "cast_null_policy": policy, "sources": split_csv(sources), "langs": split_csv(langs),
        "lang_object_path": lang_object_path, "lang_field": lang_field, "lang_map_path": lang_map_path,
        "doc_key_column": doc_key_column, "state_key": state_key,
    }
    return RunContext(
        exec_id=exec_id, run_id=str(run_id or "").strip() or exec_id, pl_name=pl_name, nb_name=NB_NAME,
        src_schema=src_schema, src_tbl=src_tbl,
        run_mode="DRY_RUN" if params["dry_run"] else "FULL_RELOAD" if params["full_reload"] else "INCREMENTAL",
        table_filter=filters, overlap_minutes=None, max_parallel=params["max_parallel"],
        max_retries=params["max_retries"], full_reload=params["full_reload"], dry_run=params["dry_run"],
        stop_on_failure=params["stop_on_failure"], running_timeout_minutes=params["running_timeout_minutes"],
        run_params=json.dumps(params, ensure_ascii=False), started_at=utc_now(),
        allow_full_scan=params["allow_full_scan"], cast_null_policy=policy,
    )


def check_config(ctx: RunContext, specs: list) -> None:
    """Kiểm tra phần cấu hình riêng của nguồn (sau load_table_specs, trước khi nhận khoá):
    mọi bảng có cột doc_key_column (entity_key của state); mọi cột raw cần đọc có trong bảng raw.
    Sai -> ConfigError, chưa ghi gì."""
    errors = [f"{s.fq_name}: không có cột {doc_key_column} (doc_key_column, dùng làm entity_key của ctrl_cdc_state)"
              for s in specs if doc_key_column not in [c.name for c in s.columns]]
    raw_cols = set(spark.table(ctx.raw_fq).columns)
    missing = [c for c in doc_columns(specs) if c not in raw_cols]
    if missing:
        errors.append(f"{ctx.raw_fq} thiếu cột {missing} (RAW_* của notebook hoặc _doc.<cột> trong registry)")
    if errors:
        raise ConfigError("Cấu hình sai:\n  - " + "\n  - ".join(errors))


def process_run(ctx: RunContext, specs: list, out: RunOutcome) -> None:
    """Phần riêng của nguồn trong 1 lần chạy (bên trong run_controlled, sau khi nhận khoá + chốt version):
    đọc raw -> phân loại + parse -> profile -> state -> 1 wave các bảng -> trạng thái -> commit."""
    t0 = time.time()
    need_langs = uses_langs(specs)
    classified = classify_docs(read_raw(ctx, doc_columns(specs)), top_fields(specs, need_langs), need_langs) \
        .persist(StorageLevel.MEMORY_AND_DISK)
    out.cached.append(classified)
    log(f"đọc raw: {ctx.read_mode} | {ctx.read_note or ''}")
    out.profile, counts, lang_stats, shapes = profile_docs(classified, specs, need_langs)
    log(f"đọc {out.profile.read_rows} | hợp lệ {out.profile.valid_rows} | bỏ qua {out.profile.ignored_rows} "
        f"| lỗi {out.profile.rejected_rows} | watermark_to {out.profile.watermark_to} ({time.time() - t0:.0f}s)")
    if lang_stats.get("missing") or lang_stats.get("not_allowed"):
        log(f"ngôn ngữ: tài liệu thiếu {lang_stats.get('missing')} | ngoài langs (bỏ qua) {lang_stats.get('not_allowed')}",
            "WARN")
    if shapes:
        log(f"khối không phải mảng JSON (bảng không nhận dòng từ tài liệu đó): {shapes}", "WARN")
    out.extra["langs"] = lang_stats
    out.extra["docs_by_table"] = counts
    out.extra["bad_shape"] = shapes

    state_df = None
    if not ctx.full_reload:
        state_df = load_state(specs).persist(StorageLevel.MEMORY_AND_DISK)
        out.cached.append(state_df)
    docs = classified.where("_status = 'VALID'")
    run_tables(specs, lambda s: counts[s.trg_tbl],
               lambda s, wave: process_table(s, docs, state_df, ctx, wave), ctx, out.results)
    out.status = decide_status(out.results)
    if not ctx.dry_run:
        t0 = time.time()
        commit(classified, out.profile, out.results, ctx)
        log(f"commit xong ({time.time() - t0:.0f}s)")


def run_extract() -> dict:
    """Luồng chính. Cấu hình sai -> raise trước khi ghi gì. Không nhận được khoá -> SKIPPED_CONCURRENT.
    Lỗi khác -> FAILED, vẫn đóng watermark, run log và nhả khoá (run_controlled)."""
    ctx = build_context()
    specs, _ = load_table_specs(ctx, SNAPSHOT_MODES)
    check_config(ctx, specs)
    load_watermark(ctx)
    log(f"{ctx.raw_fq} | mode={ctx.run_mode} | {len(specs)} bảng | last_src_version={ctx.src_version_from} "
        f"| watermark={ctx.watermark_from} | exec_id={ctx.exec_id}")
    return run_controlled(ctx, lambda out: process_run(ctx, specs, out))


RUN_SUMMARY = run_extract()

StatementMeta(, 611ec303-fa15-471c-a8d4-04a943aaf370, 22, Finished, Available, Finished, False)

[09:41:37][INFO] lh_vv_bronze.dbo.poi_raw_event | mode=INCREMENTAL | 13 bảng | last_src_version=None | watermark=None | exec_id=88ea49e7-58d5-4941-8f0b-d2bba81964a5
[09:41:50][WARN] ctrl_mng_watermark: lỗi tạm thời, thử lại sau 3s (1/5)
[09:42:03][INFO] nhận khoá (WATERMARK)
[09:42:16][INFO] mở run log, chốt version 1043 (13s)
[09:42:18][INFO] đọc raw: FULL | chưa có last_src_version (lần đầu / sau reset / sau cutover) | allow_full_scan
[09:42:45][INFO] đọc 21168 | hợp lệ 21168 | bỏ qua 0 | lỗi 0 | watermark_to 2026-09-16 11:01:11.533746 (29s)
[09:42:45][WARN] ngôn ngữ: tài liệu thiếu {'en': 8505, 'ko': 8487} | ngoài langs (bỏ qua) {}
[09:42:47][INFO][w1] priority 1: 13 việc ['slv_3p_poi', 'slv_3p_poi_address', 'slv_3p_poi_amenity', 'slv_3p_poi_contact', 'slv_3p_poi_content', 'slv_3p_poi_enrichment', 'slv_3p_poi_media', 'slv_3p_poi_opening_hours', 'slv_3p_poi_price', 'slv_3p_poi_rating', 'slv_3p_poi_raw_data', 'slv_3p_poi_review', 'slv_3p_poi_review_i18n']
[09:42:52][INFO][w1][slv_3p_p

In [8]:
# =============================================================================
# EXIT (ngoài try/except; pipeline đọc exitValue)
# =============================================================================
_summary_json = json.dumps(RUN_SUMMARY, ensure_ascii=False, default=str)
print(json.dumps(RUN_SUMMARY, ensure_ascii=False, indent=2, default=str))
if RUN_SUMMARY["status"] in ("FAILED", "PARTIAL_FAILED", "SKIPPED_CONCURRENT"):
    raise RuntimeError(f"{NB_NAME} {RUN_SUMMARY['status']}: {_summary_json}")
notebookutils.notebook.exit(_summary_json)

StatementMeta(, 611ec303-fa15-471c-a8d4-04a943aaf370, 23, Finished, Available, Finished, False)

{
  "status": "SUCCESS",
  "exec_id": "88ea49e7-58d5-4941-8f0b-d2bba81964a5",
  "run_id": "88ea49e7-58d5-4941-8f0b-d2bba81964a5",
  "run_mode": "INCREMENTAL",
  "src": "lh_vv_bronze.dbo.poi_raw_event",
  "src_version_from": null,
  "src_version": 1043,
  "read_mode": "FULL",
  "read_note": "chưa có last_src_version (lần đầu / sau reset / sau cutover) | allow_full_scan",
  "watermark_from": null,
  "watermark_to": "2026-09-16 11:01:11.533746",
  "tables": {
    "total": 13,
    "ok": 13,
    "failed": 0,
    "skipped": 0,
    "no_data": 0
  },
  "failed_tables": [],
  "rows": {
    "read": 21168,
    "valid": 21168,
    "ignored": 0,
    "rejected": 0,
    "inserted": 513751,
    "updated": 0,
    "cast_null": 0
  },
  "duration_s": 212.9,
  "error": null,
  "langs": {
    "missing": {
      "en": 8505,
      "ko": 8487
    },
    "not_allowed": {}
  },
  "docs_by_table": {
    "slv_3p_poi": 21168,
    "slv_3p_poi_address": 21168,
    "slv_3p_poi_amenity": 21168,
    "slv_3p_poi_contact

## Vận hành

**Chạy tay hoặc script ngoài FL_00** — pipeline không có Switch và không gọi notebook này. `run_id` rỗng thì notebook dùng `exec_id`.
Đặt song song với chuỗi `NB_00` cũ trong cùng nhánh (không phụ thuộc nhau) cho tới khi gold mới thay thế bảng `poi_*` cũ.
`NB_LIB_EXTRACT_RAWDATA` (bản 03/10) phải nằm cùng workspace, cùng default lakehouse (`lh_vv_bronze`).

| Việc | Tham số |
|---|---|
| Lần đầu | 1) `dry_run = True, allow_full_scan = True`: xem `cast_null` trong log từng bảng và `langs` trong exit JSON, sửa registry nếu cần; 2) `allow_full_scan = True` chạy thật; 3) bật lịch, các lần sau để `False` |
| Chạy thử, không ghi gì | `dry_run = True` |
| Nạp lại 1 vài bảng | `table_filter = "slv_3p_poi_review"`, `full_reload = True` (watermark không đổi) |
| Thêm cột | `INSERT` 1 dòng vào `ctrl_cfg_schema_registry` (notebook tự `ADD COLUMNS`); dữ liệu cũ của cột mới: chạy `full_reload` bảng đó |
| Thêm ngôn ngữ | Tham số `langs` (vd `"vi,en,ko,ja"`); tài liệu cũ: `full_reload` các bảng `LANG*` |
| Thêm nguồn crawl (vd tripadvisor) | Tham số `sources`; kiểm tra payload cùng cấu trúc trước. Tài liệu cũ của nguồn đó đã bị bỏ qua (IGNORED) ở các lần trước: chạy 1 lần `full_reload = True` |
| Exit JSON có `bad_shape` / `langs.primary_without_lang` | Payload khác cấu trúc cấu hình (khối mảng nhận object; object bản địa hoá chính thiếu `lang`): xem mẫu ở raw, sửa cấu hình hoặc báo collector |
| Bảng FAILED vì `CastNullError` | Sửa `data_type`/`convert_rule` rồi chạy lại; hoặc chấp nhận NULL: 1 lần `cast_null_policy = "WARN"` |
| `VersionGapError` / `SKIPPED_CONCURRENT` | Như partner (xem `NB_EXTRACT_PARTNER_CDC_BRZ_TO_SLV`, mục Vận hành) |

**Kiểm tra sau mỗi lần chạy**
```sql
SELECT status, run_mode, read_mode, read_rows, valid_rows, ignored_rows, rejected_rows, tbl_success_count,
       tbl_failed_count, watermark_to, duration_ms, error_message
FROM lh_vv_bronze.ctrl.ctrl_log_run WHERE src_tbl = 'poi_raw_event' ORDER BY started_at DESC LIMIT 10;

SELECT trg_tbl, status, input_rows, stale_rows, applicable_rows, entity_rows, inserted_rows, updated_rows,
       cast_null_rows, rejected_rows, duration_ms, error_message
FROM lh_vv_bronze.ctrl.ctrl_log_table_run WHERE exec_id = '<exec_id>' ORDER BY trg_tbl;

SELECT trg_tbl, reject_reason, reject_detail, COUNT(*) AS so_dong, SUM(reject_count) AS so_lan
FROM lh_vv_bronze.ctrl.ctrl_cdc_reject WHERE src_tbl = 'poi_raw_event' AND NOT is_resolved
GROUP BY 1, 2, 3 ORDER BY so_dong DESC;

-- Đối soát với bảng cũ: số địa điểm (poi_id mới = poi_group_id cũ)
SELECT (SELECT COUNT(*) FROM lh_vv_silver.dbo.slv_3p_poi) AS moi,
       (SELECT COUNT(DISTINCT poi_group_id) FROM lh_vv_silver.dbo.poi_source_map) AS cu;
```

**Bảng đích** (cấu hình ở `ctrl_mng_pipeline_config` + `ctrl_cfg_schema_registry`, seed bằng `claude/NB_CREATE_DDL.ipynb`):

| Bảng | load_mode | src_object | Khoá | Bảng cũ tương ứng |
|---|---|---|---|---|
| `slv_3p_poi` | DOC | `$` | `poi_id` | `poi_source_map` / `poi_entity` (phần danh tính nguồn) |
| `slv_3p_poi_address` | DOC | `poi_address` | `poi_id` | `poi_address` |
| `slv_3p_poi_contact` | DOC | `poi_contact` | `poi_id` | `poi_contact` |
| `slv_3p_poi_price` | DOC | `poi_price` | `poi_id` | `poi_price` |
| `slv_3p_poi_opening_hours` | DOC | `poi_opening_hours` | `poi_id` | `poi_opening_hours` |
| `slv_3p_poi_policy` | DOC | `policies` | `poi_id` | `poi_policy` — **[04/10] tắt** (`is_active = 0`, khối luôn null) |
| `slv_3p_poi_rating` | DOC | `poi_rating` | `poi_id` | `poi_rating` |
| `slv_3p_poi_amenity` | DOC | `poi_amenity` | `poi_id` | `poi_amenity_source` |
| `slv_3p_poi_raw_data` | DOC | `raw_data` | `poi_id` | (mới) |
| `slv_3p_poi_content` | DOC_ARRAY | `poi_content` | `poi_id, locale, content_type` ([SỬA 04/10] thêm `content_type`) | `poi_content` |
| `slv_3p_poi_review` | DOC_ARRAY | `poi_review` | `review_id` | `poi_review` (bản gốc) |
| `slv_3p_poi_media` | DOC_ARRAY | `poi_media` | `media_dedup_key` | `poi_media` |
| `slv_3p_poi_enrichment` | LANG | `$` | `poi_id, lang` | `poi_enrichment` |
| `slv_3p_poi_review_i18n` | LANG_ARRAY | `reviews` (align `poi_review`) | `review_id, lang` | `poi_review` (bản dịch) |
